In [0]:
%run ./99_Config

In [0]:
%run ./98_Utilities

In [0]:
%run ./97_Logger

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

pipeline_start("03_Ingestion")

# COMMAND ----------
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("account_number", StringType(), True),
    StructField("transaction_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("branch", StringType(), True),
    StructField("transaction_timestamp", TimestampType(), True)
])

# COMMAND ----------
sources = discover_source_systems()

total_sources = 0
total_files = 0
total_rows = 0

for source in sources:

    source_path = f"{INCOMING_PATH}/{source}"
    bronze_path = f"{BRONZE_PATH}/transactions"
    processed_path = f"{PROCESSED_PATH}/{source}"

    print_header(f"Processing Source : {source}")

    try:

        csv_files = get_csv_files(source_path)

        if not csv_files:
            log_warning(f"No CSV files found in {source_path}")
            continue

        df = read_csv(source_path, transaction_schema)

        rows = record_count(df)

        bronze_df = (
            df
            .withColumn("source_system", lit(source))
            .withColumn("source_file", input_file_name())
            .withColumn("pipeline_name", lit(PIPELINE_NAME))
            .withColumn("pipeline_run_id", lit(CURRENT_RUN_ID))
            .withColumn("load_date", current_date())
            .withColumn("load_timestamp", current_timestamp())
        )

        write_delta(bronze_df, bronze_path, "append")

        create_delta_table("bronze_transactions", bronze_path)

        moved = move_processed_files(source_path, processed_path)

        audit(
            notebook="03_Ingestion",
            dataset=source,
            rows_read=rows,
            rows_written=rows,
            rejected_rows=0,
            status="SUCCESS"
        )

        log_success(
            notebook="03_Ingestion",
            dataset=source,
            rows_read=rows,
            rows_written=rows,
            execution_time="Completed"
        )

        display_summary(source, rows, rows, 0)

        total_sources += 1
        total_files += moved
        total_rows += rows

    except Exception as ex:

        log_error(
            notebook="03_Ingestion",
            dataset=source,
            exception=ex
        )

        audit(
            notebook="03_Ingestion",
            dataset=source,
            rows_read=0,
            rows_written=0,
            rejected_rows=0,
            status="FAILED",
            error=str(ex)
        )


In [0]:
print("="*80)
print("INGESTION SUMMARY")
print("="*80)
print(f"Sources Processed : {total_sources}")
print(f"Files Processed   : {total_files}")
print(f"Rows Loaded       : {total_rows}")
print("="*80)

pipeline_end("03_Ingestion")
